# SQLite Database and CRUD Operations

This notebook creates the SQLite database schema for storing commodity price data and shows all basic CRUD operations.

In [15]:
import sqlite3
from pathlib import Path
import pandas as pd

## 1. Connect to the Database

We use Python's built-in `sqlite3` module. Connecting to a path that doesn't exist yet creates the file automatically.

In [ ]:
DB_PATH = Path("../data/oildesk.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

print(f"Connected to {DB_PATH}")

Connected to ../data/oildesk.db


## 2. Create the Table

The schema stores one settlement price per commodity per date.

- `id` — auto-incrementing primary key
- `date` — trading date in ISO format (YYYY-MM-DD)
- `commodity` — commodity name e.g. copper, zinc, crude_oil
- `price` — settlement price
- `source` — data source e.g. Bloomberg
- `created_at` — timestamp of when the row was inserted

The `UNIQUE` constraint on `(date, commodity)` prevents duplicate entries for the same date and commodity.

In [21]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS prices (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT NOT NULL,
        commodity TEXT NOT NULL,
        price REAL NOT NULL,
        source TEXT NOT NULL DEFAULT 'Bloomberg',
        created_at TEXT NOT NULL DEFAULT (datetime('now')),
        UNIQUE(date, commodity)
    )
""")

conn.commit()
print("Table created.")

Table created.


## 3. Insert Records

We insert a few sample rows manually here. The full pipeline insert happens in Q3.

We use `INSERT OR IGNORE` so re-running the notebook doesn't throw an error on duplicate rows.

In [22]:
sample_rows = [
    ("2021-01-04", "copper", 7833.0, "Bloomberg"),
    ("2021-01-04", "zinc", 2718.0, "Bloomberg"),
    ("2021-01-04", "crude_oil", 52.8, "Bloomberg"),
    ("2021-01-05", "copper", 7910.0, "Bloomberg"),
    ("2021-01-05", "zinc", 2730.0, "Bloomberg"),
    ("2021-01-05", "crude_oil", 53.1, "Bloomberg"),
]

cursor.executemany("""
    INSERT OR IGNORE INTO prices (date, commodity, price, source)
    VALUES (?, ?, ?, ?)
""", sample_rows)

conn.commit()
print(f"Inserted {cursor.rowcount} rows.")

Inserted 6 rows.


## 4. Read Records

Query all rows and display them as a DataFrame.

In [24]:
df = pd.read_sql("SELECT * FROM prices ORDER BY date, commodity", conn)
df

,id,date,commodity,price,source,created_at
0,1,2021-01-04,copper,7833.0,Bloomberg,2026-06-04 12:24:41
1,3,2021-01-04,crude_oil,52.8,Bloomberg,2026-06-04 12:24:41
2,2,2021-01-04,zinc,2718.0,Bloomberg,2026-06-04 12:24:41
3,4,2021-01-05,copper,7910.0,Bloomberg,2026-06-04 12:24:41
4,6,2021-01-05,crude_oil,53.1,Bloomberg,2026-06-04 12:24:41
5,5,2021-01-05,zinc,2730.0,Bloomberg,2026-06-04 12:24:41


## 5. Update a Record

Correct a price - for example if a settlement was revised.

In [25]:
cursor.execute("""
    UPDATE prices
    SET price = 7850.0
    WHERE date = '2021-01-04' AND commodity = 'copper'
""")

conn.commit()
print(f"Rows updated: {cursor.rowcount}")

# Confirm
pd.read_sql("SELECT * FROM prices WHERE date = '2021-01-04'", conn)

Rows updated: 1


,id,date,commodity,price,source,created_at
0,1,2021-01-04,copper,7850.0,Bloomberg,2026-06-04 12:24:41
1,3,2021-01-04,crude_oil,52.8,Bloomberg,2026-06-04 12:24:41
2,2,2021-01-04,zinc,2718.0,Bloomberg,2026-06-04 12:24:41


## 6. Delete a Record

Remove a specific row — for example a bad data point.

In [26]:
cursor.execute("""
    DELETE FROM prices
    WHERE date = '2021-01-05' AND commodity = 'crude_oil'
""")

conn.commit()
print(f"Rows deleted: {cursor.rowcount}")

# Confirm
pd.read_sql("SELECT * FROM prices ORDER BY date, commodity", conn)

Rows deleted: 1


,id,date,commodity,price,source,created_at
0,1,2021-01-04,copper,7850.0,Bloomberg,2026-06-04 12:24:41
1,3,2021-01-04,crude_oil,52.8,Bloomberg,2026-06-04 12:24:41
2,2,2021-01-04,zinc,2718.0,Bloomberg,2026-06-04 12:24:41
3,4,2021-01-05,copper,7910.0,Bloomberg,2026-06-04 12:24:41
4,5,2021-01-05,zinc,2730.0,Bloomberg,2026-06-04 12:24:41


## 7. Close the Connection

In [27]:
conn.close()
print("Connection closed.")

Connection closed.
